# SVD — Singular Value Decomposition (from scratch)

Every matrix hides a secret structure. SVD reveals it.

Think of any matrix as a transformation — it stretches, rotates, and flips space. SVD breaks that transformation into exactly **three simple steps**:

1. **Rotate** (Vᵀ) — align with the matrix's "natural axes"
2. **Stretch** (Σ) — scale along each axis (these are the singular values)
3. **Rotate again** (U) — orient into the output space

**A = U · Σ · Vᵀ**

| Component | Shape | What it is |
|-----------|-------|------------|
| **U** | m × m | Left singular vectors (orthonormal columns) |
| **Σ** | m × n | Diagonal matrix of singular values (σ₁ ≥ σ₂ ≥ ...) |
| **V** | n × n | Right singular vectors (orthonormal columns) |

The magical part? The singular values (Σ) tell you **how important** each direction is. The biggest one captures the most "energy." Throw away the small ones and you still keep a great approximation — that's how image compression, Netflix recommendations, and noise removal work.

## How we build it from scratch

Power iteration needs a **square** matrix, but A may not be square (e.g., 100×2). The trick:

1. Compute **AᵀA** — always square (n × n) and symmetric, even if A isn't
2. Find eigenvectors of AᵀA using **power iteration + deflation** → columns of **V**
3. Singular values: **σᵢ = √λᵢ** (square roots of eigenvalues of AᵀA)
4. Left singular vectors: **uᵢ = A · vᵢ / σᵢ** → columns of **U**

### Power Iteration
Finds the dominant eigenvector by repeatedly multiplying a random vector and normalizing:
`v ← Mv / ||Mv||` until convergence. Like a compass finding north.

### Finding the Eigenvalue (Rayleigh Quotient)
Once you have an eigenvector `v` (unit length), find its eigenvalue `λ` by projecting the matrix onto it:
`λ = vᵀ · M · v`

### Deflation
Removes the found eigenvector's contribution to find the next one:
`M_deflated = M - λᵢ · vᵢ · vᵢᵀ` — peel the matrix like an onion, layer by layer.

In [3]:
import numpy as np
import math

In [ ]:
A = np.array([[2, 1], [1, 2]])
A

array([[2, 1],
       [1, 2]])

In [ ]:
def power_iteration(M, num_iters, v):
    if num_iters == 0:
        return v

    v_new = np.dot(M, v)
    v_new = v_new / np.linalg.norm(v_new)

    if np.allclose(v, v_new):
        return v_new
    
    return power_iteration(M, num_iters - 1, v_new)

In [ ]:
e_v1 = power_iteration(A, 1000, np.random.random(2))
e_v1

array([0.70710824, 0.70710532])

In [ ]:
e_val1 = np.dot(np.dot(np.transpose(e_v1), A), e_v1)
e_val1

np.float64(2.999999999991432)

In [ ]:
A_deflated = A - e_val1 * np.outer(e_v1, np.transpose(e_v1))
A_deflated

array([[ 0.49999379, -0.5       ],
       [-0.5       ,  0.50000621]])

In [ ]:
e_v2 = power_iteration(A_deflated, 10, np.random.random(2))
e_v2

array([-0.70710239,  0.70711117])

In [ ]:
e_val2 = np.dot(np.dot(np.transpose(e_v2), A_deflated), e_v2)
e_val2

np.float64(1.0000000000257012)

In [ ]:
A_deflated_2 = A - e_val2 * np.outer(e_v2, np.transpose(e_v2))
A_deflated_2

array([[1.50000621, 1.5       ],
       [1.5       , 1.49999379]])

In [ ]:
AtA = np.array(A).T @ np.array(A)
AtA

array([[5, 4],
       [4, 5]])

In [ ]:
Ata_e_v1 = power_iteration(AtA, 10, np.random.random(2))
Ata_e_val1 = np.dot(np.dot(np.transpose(Ata_e_v1), AtA), Ata_e_v1)
Ata_e_v1, Ata_e_val1


(array([0.70710663, 0.70710693]), np.float64(8.99999999999963))

In [ ]:
Ata_deflated = AtA - Ata_e_val1 * np.outer(Ata_e_v1, np.transpose(Ata_e_v1))
Ata_deflated

array([[ 0.50000193, -0.5       ],
       [-0.5       ,  0.49999807]])

In [ ]:
Ata_e_v2 = power_iteration(Ata_deflated, 10, np.random.random(2))
Ata_e_val2 = np.dot(np.dot(np.transpose(Ata_e_v2), Ata_deflated), Ata_e_v2)
Ata_e_v2, Ata_e_val2

(array([-0.70710815,  0.70710542]), np.float64(1.0000000000033158))

In [ ]:
V = np.column_stack([Ata_e_v1, Ata_e_v2])
V

array([[ 0.70710663, -0.70710815],
       [ 0.70710693,  0.70710542]])

In [ ]:
sigma1 = math.sqrt(Ata_e_val1)
sigma2 = math.sqrt(Ata_e_val2)
sigma1, sigma2

(2.9999999999999383, 1.0000000000016578)

In [ ]:
u1 = A @ V[:, 0] / sigma1
u2 = A @ V[:, 1] / sigma2
u1, u2

(array([0.70710673, 0.70710683]), array([-0.70711088,  0.70710268]))

In [ ]:
U = np.column_stack([u1, u2])
U

array([[ 0.70710673, -0.70711088],
       [ 0.70710683,  0.70710268]])

In [ ]:
sigmas = [sigma1, sigma2]

U @ np.diag(sigmas) @ np.transpose(V)

array([[2.00000343, 0.99999828],
       [1.00000172, 1.99999657]])

Generalizing 

In [67]:
def power_iteration(matrix, v, num_iters = 1000):
	if num_iters == 0:
		return v

	v_new = np.dot(matrix, v)
	v_new = v_new / np.linalg.norm(v_new)

	if np.allclose(v, v_new):
		return v
	
	return power_iteration(matrix, v_new, num_iters - 1)

def eigen_value(matrix, v):
	return np.dot(v.T, np.dot(matrix, v))

def calculate_eigenvalues(matrix: list[list[float|int]]) -> list[float]:
	eigenvalues = []
	M = np.array(matrix)
	m, n = M.shape
	if m != n:
		raise ValueError("Input must be a square matrix")
	num = n
	while num:
		e_vec = power_iteration(M, np.random.random(n))
		e_val = eigen_value(M, e_vec)
		eigenvalues.append(round(float(e_val), 2))
		M = M - e_val * np.outer(e_vec, e_vec)
		num = num - 1

	return eigenvalues

In [68]:
calculate_eigenvalues([[2, 1], [1, 2]])

[3.0, 1.0]

In [69]:
calculate_eigenvalues([[4, -2, 1], [1, 1, 2]])

ValueError: Input must be a square matrix

In [ ]:
# implement svd(M)

In [75]:
M = [[4, -2, 1], [1, 1, 2]]
U, S, Mt = np.linalg.svd(M)
U, S, Mt

(array([[-0.9701425 , -0.24253563],
        [-0.24253563,  0.9701425 ]]),
 array([4.69041576, 2.23606798]),
 array([[-8.79049073e-01,  3.61961383e-01, -3.10252614e-01],
        [-2.14384819e-17,  6.50791373e-01,  7.59256602e-01],
        [-4.76731295e-01, -6.67423812e-01,  5.72077554e-01]]))